## Project Description

The goal of this project is accurate analysis of sentiment of tweet. The Training dataset contains different-different tweets and with respect to sentiment of tweet. We are here training our model on training data calculate the training loss and then we are evaluate our model on validation data.

### Objectives



*   **Data Preprocessing** : Cleaning and Preparing the text data for training.
*   **Tokenization** : Tokenizing the text data for model understanding.


*   **Loading Model** : Loading model for text sentiment prediction.

*   **Training** : Train our model on training data.
*   **Validation** : Evaluating model on validation text data.


### About the Dataset
This dataset contains 4 columns : tweet_id, entity, tweet_content, and sentiment. With the help of tweet_id, entity, tweet_content we have to predict sentiment of tweet and evaluate our model.

*   **Dataset** : Twitter-Sentiment-Analysis
*   **Content** : Data on various tweets text
*   **Number of rows** : 74681
*   **Number of columns** : 4



## Import the Libraries

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from google.colab import files
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, DefaultDataCollator, Trainer

In [ ]:
uploaded = files.upload()

Saving twitter_training.csv to twitter_training.csv


In [ ]:
uploaded = files.upload()

Saving twitter_validation.csv to twitter_validation.csv


## Read the Dataset

In [ ]:
df1 = pd.read_csv('twitter_training.csv')
df2 = pd.read_csv('twitter_validation.csv')

## Rename the Columns

In [ ]:
df1.columns=['tweet_id','entity','sentiment','tweet_content']
df2.columns =['tweet_id','entity','sentiment','tweet_content']

## Checking Missing Values

In [ ]:
df1.isnull().sum()

,0
tweet_id,0
entity,0
sentiment,0
tweet_content,686


In [ ]:
df2.isnull().sum()

,0
tweet_id,0
entity,0
sentiment,0
tweet_content,0


## Checking Duplicates

In [ ]:
df1.duplicated().sum()

np.int64(2700)

In [ ]:
df2.duplicated().sum()

np.int64(0)

In [ ]:
df1.drop_duplicates(inplace=True)

In [ ]:
df1 = df1[df1['tweet_content'].notnull()]

In [ ]:
df1.isnull().sum()

,0
tweet_id,0
entity,0
sentiment,0
tweet_content,0


## Data Preprocessing

In [ ]:
import re

def preprocess_text(text):
  # remove html tags
  text = re.sub(r'<.*?>','',text)

  # remove urls
  text = re.sub(r'https?://\S+','',text)

  # remove extra white space
  text = re.sub(r'\s+',' ',text).strip()

  return text

In [ ]:
df1['tweet_content'] = df1['tweet_content'].apply(preprocess_text)
df2['tweet_content'] =df2['tweet_content'].apply(preprocess_text)

In [ ]:
df1.duplicated().sum()

np.int64(412)

In [ ]:
df2.duplicated().sum()

np.int64(0)

In [ ]:
df1.drop_duplicates(inplace=True)

In [ ]:
df1.head(2)

,tweet_id,entity,sentiment,tweet_content
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...


In [ ]:
df2.head(2)

,tweet_id,entity,sentiment,tweet_content
0,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
1,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...


## Dropping Unnecessary Columns

In [ ]:
df1.drop(columns=['tweet_id','entity'],inplace=True)
df2.drop(columns=['tweet_id','entity'],inplace=True)

## Labeling the Sentiment Column

In [ ]:
le = LabelEncoder()
df1['sentiment'] = le.fit_transform(df1['sentiment'])
df2['sentiment'] = le.transform(df2['sentiment'])

## Tokenization

In [1]:
model_id = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
def tokenize_function(examples):
  return tokenizer(
      examples['tweet_content'],
      truncation=True,
      padding="max_length",
      max_length=128
  )


In [ ]:
dataset1 = Dataset.from_pandas(df1)
dataset2 = Dataset.from_pandas(df2)

In [ ]:
train_dataset = dataset1.map(tokenize_function, batched=True)
eval_dataset = dataset2.map(tokenize_function, batched=True)

## Raname Column to Labels

In [ ]:
train_dataset = train_dataset.rename_column("sentiment","labels")
eval_dataset = eval_dataset.rename_column("sentiment","labels")

## Removing the text Column

In [ ]:
train_dataset = train_dataset.remove_columns(['tweet_content'])
eval_dataset = eval_dataset.remove_columns(['tweet_content'])

## Loading a Model

In [ ]:
MODEL_NAME = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 4   # Positive, Negative, Neutral, Irrelevent
)

## Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir = "./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    # load_best_model_at_end=True,
    # metric_for_best_model="f1",
    # fp16=True,
)

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred

  predictions = np.argmax(logits, axis=-1)

  precision, recall, f1, _ = precision_recall_fscore_support(
      labels,
      predictions,
      average="weighted"
  )

  accuracy = accuracy_score(labels, predictions)

  return {
      "accuracy": accuracy,
      "precision": precision,
      "recall": recall,
      "f1": f1,
  }

In [ ]:
data_collator = DefaultDataCollator()

## Training

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.555999,0.237220,0.919920,0.922465,0.919920,0.920040
2,0.226748,0.165876,0.957958,0.958521,0.957958,0.957998
3,0.122185,0.165880,0.964965,0.965326,0.964965,0.965025


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.555999,0.237220,0.919920,0.922465,0.919920,0.920040
2,0.226748,0.165876,0.957958,0.958521,0.957958,0.957998
3,0.122185,0.165880,0.964965,0.965326,0.964965,0.965025


TrainOutput(global_step=13359, training_loss=0.3807738014516453, metrics={'train_runtime': 4870.7155, 'train_samples_per_second': 43.88, 'train_steps_per_second': 2.743, 'total_flos': 1.4058868141034496e+16, 'train_loss': 0.3807738014516453, 'epoch': 3.0})

## Evaluate Results

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.122185,0.165880,3,0.964965,0.965326,0.964965,0.965025


{'eval_loss': 0.1658802032470703, 'eval_accuracy': 0.964964964964965, 'eval_precision': 0.9653261159797786, 'eval_recall': 0.964964964964965, 'eval_f1': 0.965025271726749}


## Conclusion


*   Accuracy : 0.959
*   f1_score : 0.959
*   Training Loss : 0.170
*   Validation Loss : 0.166






In [ ]:
MODEL_PATH = "/content/bert_model"
model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/bert_model/tokenizer_config.json',
 '/content/bert_model/tokenizer.json')

In [ ]:
import os

print(os.listdir(MODEL_PATH))

['tokenizer.json', 'tokenizer_config.json', 'config.json', 'model.safetensors']


In [ ]:
import shutil

shutil.make_archive(
    "/content/bert_model",
    "zip",
    "/content/bert_model"
)

'/content/bert_model.zip'

## Downloading the Model

In [ ]:
from google.colab import files

files.download("/content/bert_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>